# Bayesian Black-Box Optimization Workflow
This notebook contains the complete logic for the BBO Capstone. 
First, we define functions for expected improvement, acquisition, and query string formatting.

In [ ]:
import numpy as np
import os
import warnings
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm
from scipy.optimize import minimize

warnings.filterwarnings('ignore')

def format_query(point):
    """
    Formats the numpy array point into the required string format:
    x1-x2-...-xn
    Each value must start with 0, have exactly six decimal places, and be separated by hyphens.
    """
    formatted_values = []
    for val in point:
        # Constraint: must be in [0.000000, 0.999999]
        val = np.clip(val, 0.0, 0.999999)
        formatted_values.append(f"{val:.6f}")
    return "-".join(formatted_values)

def expected_improvement(X, X_sample, Y_sample, gpr, xi=0.01):
    """
    Computes the EI at points X based on existing samples X_sample and Y_sample using the Gaussian process surrogate.
    Since we want to MAXIMIZE the output, we look for improvement over the maximum Y seen so far.
    """
    # X might be 1d when received from scipy.minimize, reshape to 2d
    if X.ndim == 1:
        X = X.reshape(1, -1)
    mu, sigma = gpr.predict(X, return_std=True)
    mu_sample_opt = np.max(Y_sample)

    with np.errstate(divide='warn'):
        imp = mu - mu_sample_opt - xi
        Z = imp / sigma
        ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma == 0.0] = 0.0

    return ei

def propose_next_query(acquisition_func, X_sample, Y_sample, gpr, bounds, n_restarts=10):
    """
    Proposes the next query point by optimizing the acquisition function.
    """
    dim = X_sample.shape[1]
    min_val = 1     # Want to MAXIMIZE EI, so minimize -EI
    best_x = None
    
    def objective(X):
        return -acquisition_func(X.reshape(-1, dim), X_sample, Y_sample, gpr)[0]

    for starting_point in np.random.uniform(bounds[:, 0], bounds[:, 1], size=(n_restarts, dim)):
        res = minimize(objective, 
                       x0=starting_point, 
                       bounds=bounds, 
                       method='L-BFGS-B')
        if res.fun < min_val:
            min_val = res.fun
            best_x = res.x
            
    return best_x

## Optimize All Functions
This handles checking the extracted `data/` directory and looping through all 8 available functions.
It will print out the formatted queries alongside the terminal.

In [ ]:
def optimize_function(func_id, dim, data_dir='data'):
    """
    Loads data for a specific function, fits a GP, and returns the next query string.
    """
    print(f"\n--- Optimizing Function {func_id} ---")
    func_dir = os.path.join(data_dir, f'function_{func_id}')
    inputs_path = os.path.join(func_dir, 'initial_inputs.npy')
    outputs_path = os.path.join(func_dir, 'initial_outputs.npy')
    
    if not os.path.exists(inputs_path) or not os.path.exists(outputs_path):
        print(f"[SKIPPED] Data for Function {func_id} not found at {func_dir}.")
        return None
        
    X_sample = np.load(inputs_path)
    # Flatten Y_sample to 1D array if needed
    Y_sample = np.load(outputs_path).flatten()
    
    print(f"Loaded {len(X_sample)} data points.")
    
    # Define GP Model with a Matern kernel
    kernel = Matern(nu=2.5)
    gpr = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
    
    # Fit the GP model
    gpr.fit(X_sample, Y_sample)
    
    # Define Search Space bounds
    bounds = np.array([[0.000000, 0.999999]] * dim)
    
    # Find next query
    next_query = propose_next_query(expected_improvement, X_sample, Y_sample, gpr, bounds)
    
    formatted_output = format_query(next_query)
    print(f"Next query for Function {func_id}: {formatted_output}")
    return formatted_output

# Define the dimensions for each of the 8 functions as given in the FAQs
function_dims = { 
    1: 2, 
    2: 2, 
    3: 3, 
    4: 4, 
    5: 4, 
    6: 5, 
    7: 6, 
    8: 8 
}

# Optimize all functions that have data present in the data folder
for f_id, f_dim in function_dims.items():
    optimize_function(func_id=f_id, dim=f_dim)